In [1]:
import pandas as pd
import pandas as pd
from ta import add_all_ta_features
from ta.utils import dropna
from news_utils import aggregate_news_sentiment
from insider_utils import get_rolling_insider_sentiment

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import mean_squared_error, r2_score


from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

from sklearn.linear_model import Ridge

import numpy as np 
import joblib
import os 


In [2]:
ticker = 'AMZN'

In [3]:
f = f'/Volumes/ExtremePro/AV_data/mkt_data/{ticker}.parquet'

In [ ]:
df = pd.read_parquet(f)
df = df.set_index(pd.to_datetime( df['date']) )



df = dropna(df)

# Add all ta features
ta_df = add_all_ta_features(
    df, open="open", high="high", low="low", close="close", volume="volume")

In [ ]:
df = df.set_index(pd.to_datetime( df['date']) )

/opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/ta/trend.py:1030: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  self._psar[i] = high2


In [7]:
import huggingface_hub
huggingface_hub.login()

In [ ]:

news_df = aggregate_news_sentiment(ticker=ticker, data_dir='/Volumes/ExtremePro/AV_data/news_data/', relevance_threshold=0.5)

In [ ]:
insider_df = get_rolling_insider_sentiment(ticker=ticker, data_dir='/Volumes/ExtremePro/AV_data/insider_data/', window_days=10)
insider_df = insider_df.shift(1)

In [10]:
insider_df['rolling_sentiment']

date
2003-08-14   -1.198789e+06
2003-08-15   -5.215269e+06
2003-08-18   -8.216284e+06
2003-08-19   -1.170094e+07
2003-08-20   -1.591839e+07
                  ...     
2025-11-17   -1.322963e+06
2025-11-20   -1.603280e+06
2025-11-21   -1.337077e+07
2025-11-24   -1.638996e+07
2025-12-01   -3.602239e+06
Name: rolling_sentiment, Length: 1004, dtype: float64

In [11]:
ta_df = ta_df.reindex( news_df['sentiment_score'].index)
ta_df['sentiment_score'] = news_df['sentiment_score']
ta_df['insider_sentiment'] = insider_df['rolling_sentiment'].reindex(ta_df.index).ffill(limit=20)
ta_df['insider_sentiment']= ta_df['insider_sentiment']/( ta_df['close'] * ta_df['volume'])

In [12]:
ta_df['insider_sentiment'].describe()

count    364.000000
mean      -0.048623
std        0.139501
min       -1.006199
25%       -0.001938
50%       -0.000162
75%       -0.000063
max        0.000000
Name: insider_sentiment, dtype: float64

In [13]:
ret = df['close'].pct_change()
fwd_ret = ret.shift(-1)
fwd_ret= fwd_ret.reindex(ta_df.index)
ta_df['ret'] = ret.reindex(ta_df.index)


In [14]:
ta_df['insider_sentiment'].shift(1).corr(fwd_ret)

np.float64(0.0651525865044474)

In [15]:
from labeling_utils import categorize_returns
# Example
threshold = 0.01
y = categorize_returns(fwd_ret, threshold)

In [16]:
ta_df.shape, y.shape

((439, 95), (439,))

In [17]:
a1 = ta_df['insider_sentiment']
a1[a1!=0]

date
2024-01-04   -0.000009
2024-01-08   -0.000010
2024-01-10   -0.000011
2024-01-12   -0.000012
2024-01-15         NaN
                ...   
2025-12-22   -0.000489
2025-12-23         NaN
2025-12-24         NaN
2025-12-25         NaN
2025-12-26         NaN
Name: insider_sentiment, Length: 437, dtype: float64

In [18]:
# ta_df = ta_df.iloc[252:]
# fwd_ret = fwd_ret.iloc[252:]

# y_l = y.iloc[252:]
ta_df = ta_df.iloc[:, 5:].fillna(0.0)
ta_df_diff = ta_df.fillna(0.0).diff(0).fillna(0.0)
ta_df_diff.columns = [f'{col}_diff' for col in ta_df_diff.columns]
all_ta_df = pd.concat([ta_df, ta_df_diff], axis=1)


In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(ta_df, fwd_ret.fillna(0.0), test_size=0.5, random_state=42)


In [20]:
X_train['insider_sentiment'].corr(y_train), X_test['insider_sentiment'].corr(y_test)    

(np.float64(-0.04755291440035087), np.float64(0.0033933581582354837))

In [21]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import mean_squared_error, r2_score


from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

from sklearn.linear_model import Ridge

import numpy as np 
# Load data

# Initialize a classifier
regressor = Ridge(alpha=0.1)
#regressor = TabPFNRegressor(device="cpu", ignore_pretraining_limits=True)  # Uses TabPFN 2.5 weights, finetuned on real data.
# To use TabPFN v2:
# clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
regressor.fit(X_train, y_train)


# Predict on the test set
predictions = regressor.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Squared Error (MSE):", mse)
print("R² Score:", r2)
corr = np.corrcoef(y_test, predictions)
print('correlation', corr[0,1]  )

Mean Squared Error (MSE): 0.0004804679785573973
R² Score: -0.341544986279533
correlation 0.12202956153605601


/opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=2.46428e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


In [22]:
w = np.sign(predictions)
p = w * y_test
p.mean() * 16 / p.std()

np.float64(0.7789719573425233)

In [2]:
def train_model(ticker, dir='/Volumes/ExtremePro/AV_data/models/linear/'):
    os.makedirs(dir, exist_ok=True)
    try:
        f = f'/Volumes/ExtremePro/AV_data/mkt_data/{ticker}.parquet'
        df = pd.read_parquet(f)
        df = df.set_index(pd.to_datetime( df['date']) )
    
        df = dropna(df)

        ta_df = add_all_ta_features(
        df, open="open", high="high", low="low", close="close", volume="volume")
        news_df = aggregate_news_sentiment(ticker=ticker, data_dir='/Volumes/ExtremePro/AV_data/news_data/', relevance_threshold=0.5)
        insider_df = get_rolling_insider_sentiment(ticker=ticker, data_dir='/Volumes/ExtremePro/AV_data/insider_data/', window_days=10)
        insider_df = insider_df.shift(1)

        ta_df = ta_df.reindex( news_df['sentiment_score'].index)
        ta_df['sentiment_score'] = news_df['sentiment_score']
        ta_df['insider_sentiment'] = insider_df['rolling_sentiment'].reindex(ta_df.index).ffill(limit=20)
        ta_df['insider_sentiment']= ta_df['insider_sentiment']/( ta_df['close'] * ta_df['volume'])

        ret = df['close'].pct_change()
        fwd_ret = ret.shift(-1)
        fwd_ret= fwd_ret.reindex(ta_df.index)
        ta_df['ret'] = ret.reindex(ta_df.index)

        regressor = Ridge(alpha=0.1)

        regressor.fit(ta_df.fillna(0.0).iloc[:,6:], fwd_ret.fillna(0.0))

        f = f'{dir}/{ticker}.joblib'
        joblib.dump(regressor, f)

        return True 
    except Exception as e :
        print(f'{ticker} error {e}')
        return False    












In [1]:
from sp500_utils import get_sp500_tickers
tickers = get_sp500_tickers()



In [ ]:
for ticker in tickers:
    print(ticker)
    train_model(ticker, dir='/Volumes/ExtremePro/AV_data/models/linear/')
